#Evaluation gegen Benchmark

Benchmark-Datensatz wird klassifizerit und mit den vorhandenen Labels verglichen.

In [ ]:
import pandas as pd
import re
from vaderSentiment.vaderSentiment import SentimentIntensityAnalyzer
from sklearn.metrics import classification_report, confusion_matrix

analyzer = SentimentIntensityAnalyzer()

# Auf False setzen, um Ansatz 2 (Transformer) zu überspringen, falls der Rechner/die
# Internetverbindung dafür gerade nicht ausreicht...Ansatz 1 läuft dann trotzdem komplett durch. Abeer würde empfehlen das sowieso über Colab zu machen.
TRANSFORMER = True

In [ ]:
def bereinige_text(text):
    text = str(text).lower()
    text = re.sub('\\[.*?\\]', ' ', text)
    text = re.sub('[^a-z0-9\\s]', ' ', text)
    text = re.sub('\\s+', ' ', text).strip()
    return text

#Risiko-Klassifikation: Ansatz 1 (regelbasiert)

Dieselben Schlüsselwortlisten wie in `05_risiko_klassifikation.ipynb`. Wenn kein Schlüsselwort greift, entscheidet der Vader-compound-Wert: liegt er unter -0.3, wird es  als ästhetische Melancholie gewertet.

In [ ]:
SCHLÜSSELWOERTER_RISIKOHINWEIS = ['hopeless', 'no way out', 'give up', 'give in', 'no strength left', 'disappear', 'nobody hears me', 'pointless', 'the end', 'cant go on', "can't go on", 'worthless']
SCHLÜSSELWOERTER_HERZSCHMERZ = ['heartbreak', 'miss you', "you're gone", 'youre gone', 'broken heart', 'left me', 'lonely without you', 'without you']
SCHLÜSSELWOERTER_NOSTALGIE = ['i remember', 'back then', 'old photos', 'those days', 'nostalgia', 'used to']

def enthält_schlüsselwort(text, wörter_liste):
    return any((wort in text for wort in wörter_liste))

def klassifiziere_risiko_ansatz1(text):
    if enthält_schlüsselwort(text, SCHLÜSSELWOERTER_RISIKOHINWEIS):
        return 'möglicher_risikohinweis'
    if enthält_schlüsselwort(text, SCHLÜSSELWOERTER_HERZSCHMERZ):
        return 'herzschmerz'
    if enthält_schlüsselwort(text, SCHLÜSSELWOERTER_NOSTALGIE):
        return 'ästhetische_melancholie'
    werte = analyzer.polarity_scores(text)
    if werte['compound'] <= -0.3:
        return 'ästhetische_melancholie'
    return 'kein_hinweis'

#Risiko-Klassifikation: Ansatz 2 (mit Transformer-Signal)

Ansatz 2 nutzt dieselben Schlüsselwörter wie Ansatz 1, ergänzt sie aber um das Signal aus dem GoEmotions-Transformer (04). Wenn kein Schlüsselwort greift, aber die stärkste erkannte Emotion eine "traurige" ist (sadness/grief/disappointment/remorse) und Vader die Zeile insgesamt negativ einordnet, wird das ebenfalls als ästhetische Melancholie gewrtet. Das ist  dieselbe Logik wie in 05. Hier wird also getestet, ob das Hinzunehmen des Transformer-Signals die Übereinstimmung mit dem externen Benchmark verbessert oder verschlechtert, im Vergleich zu dem rein regelbasiertem Ansatz 1.

In [ ]:
MODELL_NAME = 'SamLowe/roberta-base-go_emotions'
EMOTIONEN_TRAURIG = ['sadness', 'grief', 'disappointment', 'remorse']

def lade_emotions_modell():
    from transformers import pipeline
    emotion_klassifikator = pipeline(task='text-classification', model=MODELL_NAME, top_k=None)
    return emotion_klassifikator

def erkenne_emotionen(text, klassifikator, max_zeichen=512):
    gekürzter_text = text[:max_zeichen]
    ergebnis = klassifikator(gekuerzter_text)
    emotionen = ergebnis[0]
    return emotionen

def hole_top_emotion(emotionen):
    top = emotionen[0]
    for emotion in emotionen:
        if emotion['score'] > top['score']:
            top = emotion
    return (top['label'], top['score'])

def klassifiziere_compound(compound_wert):
    if compound_wert >= 0.05:
        return 'positiv'
    elif compound_wert <= -0.05:
        return 'negativ'
    else:
        return 'neutral'

def klassifiziere_risiko_ansatz2(text, top_emotion=None, vader_kategorie=None):
    if enthält_schlüsselwort(text, SCHLÜSSELWÖRTER_RISIKOHINWEIS):
        return 'möglicher_risikohinweis'
    if enthält_schlüsselwort(text, SCHLÜSSELWÖRTER_HERZSCHMERZ):
        return 'herzschmerz'
    if enthält_schlüsselwort(text, SCHLÜSSELWÖRTER_NOSTALGIE):
        return 'ästhetische_melancholie'
    if top_emotion in EMOTIONEN_TRAURIG and vader_kategorie == 'negativ':
        return 'ästhetische_melancholie'
    return 'kein_hinweis'

In [ ]:
def risiko_label_zu_grob(label):
    if label == 'möglicher_risikohinweis':
        return 'möglicherweise_auffällig'
    return 'eher_unauffällig'

def quadrant_zu_grobem_label(quadrant):
    zuordnung = {'Q1': 'eher_unauffällig', 'Q2': 'eher_unauffällig', 'Q3': 'möglicherweise_auffällig', 'Q4': 'eher_unauffällig'}
    return zuordnung.get(quadrant, 'unbekannt')

#Pipeline ausführen

In [ ]:
benchmark = pd.read_csv('data/benchmark_lyrics_mit_labels.csv')
print(f'{len(benchmark)} Songs mit Songtexten und Benchmark-Labels geladen.')
benchmark.head()

In [ ]:
benchmark['lyrics_clean'] = benchmark['lyrics'].apply(bereinige_text)
benchmark['benchmark_grob'] = benchmark['quadrant'].apply(quadrant_zu_grobem_label)

#Ansatz 1: regelbasiert (Vader + Keywords)
benchmark['risiko_label_ansatz1'] = benchmark['lyrics_clean'].apply(klassifiziere_risiko_ansatz1)
benchmark['risiko_grob_ansatz1'] = benchmark['risiko_label_ansatz1'].apply(risiko_label_zu_grob)

benchmark['vader_compound'] = benchmark['lyrics_clean'].apply(lambda t: analyzer.polarity_scores(t)['compound'])
benchmark['vader_kategorie'] = benchmark['vader_compound'].apply(klassifiziere_compound)

# Ansatz 2: Keywords + Transformer-Signal
if TRANSFORMER:
    print('Lade Transformer-Modell ...')
    klassifikator = lade_emotions_modell()
    top_emotion_liste = []
    for text in benchmark['lyrics_clean']:
        emotionen = erkenne_emotionen(text, klassifikator)
        top_label, top_score = hole_top_emotion(emotionen)
        top_emotion_liste.append(top_label)
    benchmark['top_emotion'] = top_emotion_liste
    benchmark['risiko_label_ansatz2'] = benchmark.apply(lambda zeile: klassifiziere_risiko_ansatz2(zeile['lyrics_clean'], zeile['top_emotion'], zeile['vader_kategorie']), axis=1)
    benchmark['risiko_grob_ansatz2'] = benchmark['risiko_label_ansatz2'].apply(risiko_label_zu_grob)
else:
    print('TRANSFORMER = False -> Ansatz 2 wird übersprungen, nur Ansatz 1 wird ausgewertet.')

print(benchmark[['title', 'artist', 'quadrant', 'benchmark_grob', 'risiko_label_ansatz1', 'risiko_grob_ansatz1']])

#Evaluation

In [ ]:
print('Ansatz 1 (Vader + Keywords) vs. Benchmark-Labels')
print(classification_report(benchmark['benchmark_grob'], benchmark['risiko_grob_ansatz1'], zero_division=0))
kategorien = sorted(set(benchmark['benchmark_grob']) | set(benchmark['risiko_grob_ansatz1']))
matrix1 = confusion_matrix(benchmark['benchmark_grob'], benchmark['risiko_grob_ansatz1'], labels=kategorien)
print(pd.DataFrame(matrix1, index=kategorien, columns=kategorien))

if TRANSFORMER:
    print('Ansatz 2 (Keywords + Transformer-Signal) vs. Benchmark-Labels')
    print(classification_report(benchmark['benchmark_grob'], benchmark['risiko_grob_ansatz2'], zero_division=0))
    kategorien2 = sorted(set(benchmark['benchmark_grob']) | set(benchmark['risiko_grob_ansatz2']))
    matrix2 = confusion_matrix(benchmark['benchmark_grob'], benchmark['risiko_grob_ansatz2'], labels=kategorien2)
    print(pd.DataFrame(matrix2, index=kategorien2, columns=kategorien2))

In [ ]:
benchmark.to_csv('data/benchmark_evaluation_ergebnis.csv', index=False)
print('\nGespeichert unter: data/benchmark_evaluation_ergebnis.csv')